# 04 Embeddings, semantic axes, decoding, BERTopic
Registered plan, Sections 7.3, 7.4 (effect size), and 7.6. Use a GPU runtime (Colab Pro+) for `Qwen/Qwen3-Embedding-8B`. API keys come from Colab Secrets: `GEMINI_API_KEY`, `VOYAGE_API_KEY`.

*Generated by `_build_notebooks.py`; edit that file, not this notebook.*

In [1]:
# --- Setup -------------------------------------------------------------
# MODE = "simulated" runs on the synthetic data (stand-ins replace the GPU models
# and APIs). MODE = "real" runs on the de-identified data. Until
# Analysis/text_frozen/text_config.json has frozen = true, real mode processes
# only the development set (plan, Section 7.1).
MODE = "simulated"
REPO_URL = "https://github.com/DGK-Law-and-Cognition-Lab/Moral-Authority-in-Copyright.git"
DRIVE_REPO = "/content/drive/MyDrive/Moral-Authority-in-Copyright"   # where the repo lives on Drive in Colab

import os, sys, json, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists(DRIVE_REPO):
        subprocess.run(["git", "clone", REPO_URL, DRIVE_REPO], check=True)
    os.chdir(DRIVE_REPO)
sys.path.insert(0, os.path.join(os.getcwd() if IN_COLAB else os.path.abspath(".."), "python"))
sys.path.insert(0, os.path.abspath(os.path.join("Analysis", "python")))
import textlib as tl
ROOT = tl.repo_root()
CFG = tl.load_config(ROOT)
assert MODE in ("simulated", "real")
SEED = CFG["seed"]
print("Repository folder:", os.path.basename(ROOT), "| mode:", MODE, "| text config frozen:", CFG["frozen"])


Repository folder: Moral-Authority-in-Copyright | mode: simulated | text config frozen: False


In [2]:
if IN_COLAB and MODE == "real":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "google-genai", "voyageai", "bertopic", "umap-learn", "hdbscan", "pyarrow"], check=True)

In [3]:
import numpy as np, pandas as pd
out = tl.output_dir(ROOT, MODE)
mt = pd.read_csv(os.path.join(out, "masked_texts.csv"), dtype={"pid": str}).fillna("")
ti = tl.load_text_input(ROOT, MODE).set_index("pid")
mt = mt.join(ti[["artist", "victim", "transform", "analyzed"]], on="pid")
print(len(mt), "texts")

545 texts


## Embedding functions
In simulated mode every model is replaced by a hashing stand-in (`tl.hash_embed`), labeled as such in the output. It is not a language model.

In [4]:
def get_secret(name):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    return os.environ.get(name)

def embed(model_cfg, texts):
    if MODE == "simulated":
        import hashlib
        return tl.hash_embed(texts, 256, seed=int(hashlib.sha256(model_cfg["key"].encode()).hexdigest(), 16) % 10000)
    p = model_cfg["provider"]
    if p == "sentence-transformers":
        import torch
        from sentence_transformers import SentenceTransformer
        if MODE == "real" and CFG["frozen"]:
            assert model_cfg.get("revision"), f"{model_cfg['id']}: revision must be pinned before the test set"
        m = SentenceTransformer(model_cfg["id"], revision=model_cfg.get("revision"),
                                model_kwargs={"torch_dtype": torch.bfloat16} if "Qwen" in model_cfg["id"] else {})
        return m.encode(list(texts), batch_size=16, normalize_embeddings=True, show_progress_bar=True)
    if p == "google":
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=get_secret("GEMINI_API_KEY"))
        vecs = []
        for i in range(0, len(texts), 50):
            r = client.models.embed_content(model=model_cfg["id"], contents=list(texts[i:i + 50]),
                                            config=types.EmbedContentConfig(task_type=model_cfg["task_type"]))
            vecs += [e.values for e in r.embeddings]
        return tl.l2_normalize(np.array(vecs))
    if p == "voyage":
        import voyageai
        vo = voyageai.Client(api_key=get_secret("VOYAGE_API_KEY"))
        vecs = []
        for i in range(0, len(texts), 64):
            vecs += vo.embed(list(texts[i:i + 64]), model=model_cfg["id"], input_type=model_cfg["input_type"]).embeddings
        return tl.l2_normalize(np.array(vecs))
    raise ValueError(p)

In [5]:
ecfg = CFG["embedding_models"]
models = [ecfg["primary"]] + ecfg["checks"]
emb_dir = tl.output_dir(ROOT, MODE, "embeddings")
E = {}
for m in models:
    for version in ("raw", "masked"):
        texts = mt["Explain_Ruling"] if version == "raw" else mt["masked"]
        X = np.asarray(embed(m, texts.tolist()), dtype=np.float32)
        E[(m["key"], version)] = X
        frame = pd.DataFrame(X, columns=[f"e{i}" for i in range(X.shape[1])])
        frame.insert(0, "pid", mt["pid"].values)
        frame.to_parquet(os.path.join(emb_dir, f"{m['key']}__{version}.parquet"), index=False)
        print(m["key"], version, X.shape, "(SIMULATED stand-in)" if MODE == "simulated" else "")

qwen3_8b raw (545, 256) (SIMULATED stand-in)
qwen3_8b masked (545, 256) (SIMULATED stand-in)


gemini2 raw (545, 256) (SIMULATED stand-in)
gemini2 masked (545, 256) (SIMULATED stand-in)
voyage4l raw (545, 256) (SIMULATED stand-in)


voyage4l masked (545, 256) (SIMULATED stand-in)
mpnet raw (545, 256) (SIMULATED stand-in)


mpnet masked (545, 256) (SIMULATED stand-in)


## Semantic axes (7.6)
Score = cos(response, mean of pole A anchors) - cos(response, mean of pole B anchors), on the masked text, for every model.

In [6]:
anchors = tl.load_anchors(ROOT)
if MODE == "real" and CFG["frozen"]:
    assert anchors["frozen"], "anchors.json must be frozen before the test set"
rows = {"pid": mt["pid"].values}
for m in models:
    for ax, poles in anchors["axes"].items():
        A = np.asarray(embed(m, poles["pole_a"])); B = np.asarray(embed(m, poles["pole_b"]))
        rows[f"ddr_{ax}__{m['key']}"] = tl.ddr_scores(E[(m["key"], "masked")], A, B)
pd.DataFrame(rows).to_csv(os.path.join(out, "ddr_scores.csv"), index=False)
print("ddr_scores.csv written")

ddr_scores.csv written


## Decoding classifier (7.4, effect size for T1-T3)
Test set only, masked text. Cross-validated L2 logistic regression (10 folds x 10 repeats) and a permutation test.

In [7]:
dc = CFG["decoding"]
n_perm = dc["n_permutations"] if MODE == "real" else 100   # fewer permutations in simulated mode, for speed
test = (mt.text_set == "test") & mt.analyzed.astype(bool)
res = []
if test.sum() > 0:
    for m in models:
        X = E[(m["key"], "masked")][test.values]
        for f in ("artist", "victim", "transform"):
            y = mt.loc[test, f].values
            r = tl.decoding_accuracy(X, y, dc["folds"], dc["repeats"], n_perm, dc["C"], SEED)
            res.append({"model": m["key"], "factor": f, **r})
pd.DataFrame(res).to_csv(os.path.join(out, "decoding_T1_T3.csv"), index=False)
pd.DataFrame(res)

,model,factor,accuracy,accuracy_sd_repeats,chance,permutation_observed_accuracy,permutation_mean,permutation_p,n,folds,repeats,n_permutations
0,qwen3_8b,artist,0.5684,0.017633,0.500,0.568,0.50784,0.118812,250,10,10,100
1,qwen3_8b,victim,0.6768,0.018165,0.504,0.648,0.50328,0.009901,250,10,10,100
2,qwen3_8b,transform,0.7716,0.013525,0.516,0.780,0.50700,0.009901,250,10,10,100
3,gemini2,artist,0.5700,0.016357,0.500,0.564,0.50688,0.108911,250,10,10,100
4,gemini2,victim,0.6936,0.018494,0.504,0.672,0.49944,0.009901,250,10,10,100
5,gemini2,transform,0.7568,0.006746,0.516,0.768,0.50344,0.009901,250,10,10,100
6,voyage4l,artist,0.5604,0.013125,0.500,0.548,0.50772,0.207921,250,10,10,100
7,voyage4l,victim,0.6716,0.017122,0.504,0.644,0.50236,0.009901,250,10,10,100
8,voyage4l,transform,0.7644,0.006915,0.516,0.768,0.50392,0.009901,250,10,10,100
9,mpnet,artist,0.5448,0.017968,0.500,0.552,0.50728,0.168317,250,10,10,100


## BERTopic (7.6)
Primary embeddings of the masked text, all explanations; 10 seeds; stability as mean adjusted Rand index.

In [8]:
bc = CFG["bertopic"]
X = E[(ecfg["primary"]["key"], "masked")]
assign = []
for s in bc["seeds"]:
    a, tm = tl.bertopic_assignments(mt["masked"].tolist(), X, s, bc["min_topic_size"])
    assign.append(a)
    if s == bc["seeds"][0]:
        tm.get_topic_info().to_csv(os.path.join(out, "bertopic_topic_info.csv"), index=False)
pd.DataFrame({"pid": mt["pid"], **{f"seed_{s}": a for s, a in zip(bc["seeds"], assign)}}).to_csv(
    os.path.join(out, "bertopic_assignments.csv"), index=False)
stab = tl.topic_stability(assign)
json.dump(stab, open(os.path.join(out, "bertopic_stability.json"), "w"), indent=2)
stab

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


{'mean_ari': 0.8727398191809244,
 'min_ari': 0.7774778335372062,
 'max_ari': 0.946189575148643,
 'pairs': 45}